# Navigation that helps some patients and not others: who should get it, and what a lottery achieves

## At a glance

A hospital can offer navigation to only some of the patients it sends home. Navigation helps
patients with high need, and it does slightly worse than usual care for patients with low need.
Who should get it, and how many patients would stay home for 30 days under a rule learned from
the program's own records? And if staff can cover only some of the patients who qualify, what
does a lottery among them achieve? Nobody randomized the offer.

Under a rule learned from these data, 71.7% of patients would stay home for 30 days (95%
confidence interval 69.1% to 74.4%). That number is the average of what ten rules achieve, each
learned on nine tenths of the patients and scored on the other tenth, not the value of one final
rule. A lottery that offers navigation to half of the patients with a need score above 0 raises
the share by 2.6 per 100 patients against offering it to no one (95% CI 1.1 to 4.2). Both are causal only if the recorded need score and living situation capture every
common cause of the offer and of staying home, which the data cannot check.

The data are simulated, and each interval contains its true value (71.7 and 3.3 per 100).

## The question

At the discharge-home order, the program decides whether to offer a patient navigation: a
navigator who helps with medicines, appointments and warning signs. The outcome is whether the
patient stays home, with no unplanned readmission, for 30 days. A treatment rule is a recipe that
decides from a patient's recorded characteristics who gets the offer, and its value is the share
who would stay home had every patient been offered navigation by that rule.

```text
  need, lives_alone ──▶ offer ──▶ stays home for 30 days
        └──────────────────────────────▲      need also changes how much the offer helps
  assumed absent: an unrecorded cause of both the offer and staying home
```

Staff offered navigation more often to patients with higher need and to patients who live alone,
and both characteristics also change the chance of staying home. They are confounders, variables
that drive both the offer and the outcome. Need does one more thing: it changes how much the offer
helps, so the best rule offers navigation to some patients and not to others.

## Set up and load the data

Every fit sets its seeds, so reruns match.

In [1]:
from dataclasses import replace

import numpy as np
import pandas as pd

import cleverly

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)
print("cleverly", cleverly.__version__)

cleverly 0.1.6


Another version of `cleverly` can print slightly different numbers.

`make_treatment_rules` draws 2,000 simulated patients and also returns the true answers, which
real data never give you.

In [2]:
from cleverly.datasets import make_treatment_rules

# With your own data, load one row per patient, such as pd.read_csv(...), with the recorded
# characteristics, the offer (0 or 1), and the outcome (1 if the patient stayed home 30 days).
# Real data have no `truth`: delete the lines marked "simulation only" here and below.
frame, truth = make_treatment_rules(n=2_000, seed=7)
print("rows and columns:", frame.shape)
print(list(frame.columns))
print(frame.head().round(2).to_string())
print("offered:", int(frame["offer"].sum()), " stayed home 30 days:", int(frame["home_30"].sum()))
print()

# simulation only, to the end of the cell
print("true share who stay home for 30 days:")
print(f"  offer to no one     {truth['ey_none']:.4f}")
print(f"  offer to everyone   {truth['ey_all']:.4f}")
best, cutoff = truth["ey_best_rule"], truth["best_rule_cutoff"]
print(f"  the best rule       {best:.4f}, offers when need > {cutoff:g}")

rows and columns: (2000, 4)
['need', 'lives_alone', 'offer', 'home_30']
   need  lives_alone  offer  home_30
0  0.25          0.0    0.0      1.0
1  0.79          0.0    1.0      1.0
2  0.55          1.0    1.0      1.0
3 -0.55          1.0    1.0      1.0
4 -0.40          1.0    1.0      1.0
offered: 1061  stayed home 30 days: 1339

true share who stay home for 30 days:
  offer to no one     0.6522
  offer to everyone   0.6763
  the best rule       0.7197, offers when need > -0.1


Each row is one patient. `need` is a need score from -1 (lowest) to 1 (highest), and
`lives_alone` is 1 for a patient who lives alone. Offering navigation to everyone would raise the
share who stay home only from 0.6522 to 0.6763, because the offer helps some patients and harms
others. The best rule, which offers navigation only where it helps, would reach 0.7197.

## Why a simple comparison misleads

In [3]:
print("average need and share living alone, by the offer:")
print(frame.groupby("offer")[["need", "lives_alone"]].mean().round(3))
print()

# The simple comparison: the share who stayed home, offered minus not offered.
shares = frame.groupby("offer")["home_30"].mean()
print(f"simple difference, everyone: {shares[1] - shares[0]:+.4f}")
print(
    f"true difference, offering to everyone against no one: {truth['ate']:+.4f}"
)  # simulation only
half = np.where(frame["need"] > 0, "need above 0", "need 0 or below")
by_half = frame.groupby([half, "offer"])["home_30"].mean().unstack()
print("simple difference by half:", (by_half[1.0] - by_half[0.0]).round(4).to_dict())

average need and share living alone, by the offer:
        need  lives_alone
offer                    
0.0   -0.096        0.473
1.0    0.084        0.524

simple difference, everyone: -0.0328
true difference, offering to everyone against no one: +0.0241
simple difference by half: {'need 0 or below': -0.1467, 'need above 0': 0.1066}


Offered patients had higher need and lived alone more often, and both lower the chance of staying
home. So the simple difference, -0.0328, says navigation hurts, while offering it to everyone
would in truth raise the share by 0.0241. Split at a need score of 0, the simple differences point
opposite ways, -0.1467 and 0.1066: a first sign that the offer helps some patients and not others.
Each half still mixes the offer's effect with who got it.

## Describe the study and its assumptions

First, describe the study (`StudyProtocol`): who is in it, when follow-up starts, what each
option means, and what the outcome is. The description does not change the estimate; it records
what the estimate means, and each result keeps a copy of it. Passing it is optional. The code
adapts the program's [shared study description](index.md#the-shared-study-design).

In [4]:
from cleverly.datasets import navigation_protocol

# For your own study, build cleverly.StudyProtocol(...) directly, with the fields printed below.
program = navigation_protocol()
protocol = replace(
    program,
    treatment_strategies=(
        *program.treatment_strategies,
        "Offer navigation by a rule learned from the recorded need and living situation",
        "Offer navigation by lottery, with chance one half, to patients with a need score above 0",
    ),
    treatment_versions=(
        *program.treatment_versions,
        "The standard offer for the patients the rule selects, usual support for the others",
        "The standard offer for the patients the lottery selects, usual support for the others",
    ),
    outcome="Stayed home for 30 days after discharge, with no unplanned readmission",
    intercurrent_event_handling=(
        "Count an unplanned readmission within 30 days as not staying home",
        program.intercurrent_event_handling[1],
        "Count death before day 30 as not staying home (composite strategy)",
    ),
)
print("\n".join(protocol.summary_lines()))

Population: Adults with a discharge-home order at a participating hospital during the enrollment period
Eligibility: Age 18 years or older; Discharge home ordered at a participating hospital
Time zero: Discharge-home order, after baseline measurement and before the navigation offer
Strategies: Offer standard transition navigation; Provide usual discharge support; Offer navigation by a rule learned from the recorded need and living situation; Offer navigation by lottery, with chance one half, to patients with a need score above 0
Versions: Bedside transition plan and two scheduled navigator contacts within 30 days; No access to the transition-navigation offer; The standard offer for the patients the rule selects, usual support for the others; The standard offer for the patients the lottery selects, usual support for the others
Outcome: Stayed home for 30 days after discharge, with no unplanned readmission
Horizon: 30 days after discharge
Intercurrent events: Count an unplanned readmissi

It adds the learned rule and the lottery to the two options of the shared description, and it
changes the outcome and the intercurrent events, the rules for events after discharge that change
what the outcome means. A readmission or a death counts as not staying home, so an option cannot
look better because its patients die first.

## Name the columns and the question

Tell `cleverly` which column holds the offer, which holds the outcome, and which characteristics
to adjust for (`PointTreatment`). `LearnedRuleValue` asks for the value of a rule that the fit
learns from the data. `identify` checks that the columns can answer that question and lists the
conditions the answer needs.

In [5]:
from cleverly import CausalStudy, LearnedRuleValue, PointTreatment

# For your own data, change the column names.
design = PointTreatment(outcome="home_30", treatment="offer", adjustment=("need", "lives_alone"))
study = CausalStudy(frame, design=design, protocol=protocol)
question = study.identify(LearnedRuleValue())
print(question.summary())

                      Causal question (ey_learned_rule)
Question:      average over the outer   Rows:                            2,000
               folds of the value of    Answered by:  adjusting for covariates
               the rule learned on
               each fold's training
               rows
Outcome:       home_30 (binary)
Treatment:     offer (levels 0, 1)
Adjusted for:  need, lives_alone
Conditions the answer rests on
condition                checkable   what it says
------------------------------------------------------------------------------
well-defined treatment   no          Each unit's observed outcome is the
                                     outcome it would have had under the
                                     treatment it actually received. This
                                     needs the treatment to be well defined:
                                     every way of getting the same level has
                                     the same effect.
no inte

`Question:` restates the target: the average value of several rules, each learned on part of the
data, as the next section explains. The three conditions, in plain words:

- well-defined treatment (called consistency in the study description): an offer means the same
  contacts for every patient;
- no interference: one patient's offer does not change another patient's chance of staying home;
- no hidden confounding: need and living alone hold every common cause of the offer and of
  staying home.

The data cannot check any of them (`checkable` reads `no`). Note [1] points to seven more
conditions in `summary(detail="full")`. The two that matter most here are overlap, which the
weight check below partly checks, and a clear best rule: the offer's effect must be exactly zero
for almost no patient. The simulation meets both. Only TMLE, explained next, answers this question.

## Estimate the value of a learned rule

The fit splits the patients into ten folds, groups of 200. For each fold, it fits an outcome
model on the other nine folds, and the rule offers navigation to a patient when that model
predicts a better chance of staying home with the offer than without it. It then estimates how
that rule does on the fold it did not learn from, with targeted maximum likelihood estimation
(TMLE): a correction step that uses the treatment model to remove the bias that the outcome model
leaves. The reported value is the average of the ten fold results.

- `outcome_learner` predicts staying home from the offer and the characteristics. The rule can
  only offer navigation by need if the model lets the offer's effect change with need, so it
  includes every product of two inputs, such as offer times need.
- `treatment_learner` predicts who was offered.
- `CrossFitting(n_folds=10, fold_evaluation=True)` learns each fold's rule without that fold and
  scores it on that fold, so no patient helps choose the rule that is then scored on them.

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures

from cleverly import CrossFitting, ModelSpec, Runtime, TMLEMethod

# For your own data, give the outcome model the offer-by-characteristic terms that the rule can
# use. C=1e6 turns off scikit-learn's default shrinkage of the coefficients.
outcome_model = make_pipeline(
    PolynomialFeatures(2, interaction_only=True, include_bias=False),
    LogisticRegression(C=1e6, max_iter=5000),
)
method = TMLEMethod(
    models=ModelSpec(
        outcome_learner=outcome_model,
        treatment_learner=LogisticRegression(C=1e6, max_iter=1000),
    ),
    cross_fitting=CrossFitting(n_folds=10, fold_evaluation=True),
    runtime=Runtime(random_state=13, n_jobs=1),
)
learned = question.estimate(method=method)
print(learned.summary())

                            Effect estimate (TMLE)
Question:      value of a treatment     Rows:                            2,000
               rule learned from the    Treated share:                   0.530
               data                     Out-of-fold predictions:      10 folds
Outcome:       home_30 (binary)         Outcome model:     logistic regression
Adjusted for:  need, lives_alone                        after polynomial terms
Method:        targeted maximum         Treatment model:   logistic regression
               likelihood (TMLE)        Probability limits:  [0.01471, 0.9853]
                               estimate  std. error          95% CI  p-value
------------------------------------------------------------------------------
ey_learned_rule[learned rule]     0.717       0.014  [0.691, 0.744]  <0.0001


The estimate row, `ey_learned_rule[learned rule]`, is the value of the learned rule: 0.717, so
about 72 of every 100 patients would stay home for 30 days (95% CI 0.691 to 0.744). The p-value
tests only whether the share is zero. `Treated share: 0.530` is the share offered in the data.
`Probability limits` sets a floor and a ceiling for each predicted chance of the offer, chosen from
the number of rows, and no patient reached them.

In [7]:
from cleverly.datasets import learned_rule_truth

value = learned["ey_learned_rule[learned rule]"]
record = learned.extra["learned_rule"]
low, high = value.ci
print(f"value of the learned rule: {value.psi:.4f}, 95% CI {low:.4f} to {high:.4f}")
print(
    f"per 100 patients: {100 * value.psi:.1f} stay home, 95% CI {100 * low:.1f} to {100 * high:.1f}"
)
fold_shares = record.treated_shares
print(
    f"share of patients each fold's rule offers: {min(fold_shares):.3f} to {max(fold_shares):.3f}"
)
# The rule each patient got from the fold that held them out: 1 for an offer.
offered_by_rule = pd.Series(learned.nuisance.regimes.values[:, 1, 0], index=frame.index)
edges, labels = [-1.0, -0.5, 0.0, 0.5, 1.0], ["-1 to -0.5", "-0.5 to 0", "0 to 0.5", "0.5 to 1"]
bands = pd.cut(frame["need"], edges, labels=labels, include_lowest=True).rename("need score")
print("share the rules offer, by need score:")
print(offered_by_rule.groupby(bands, observed=True).mean().round(3).to_string())
print()

# simulation only, to the end of the cell: the true value of the ten fold rules, which the
# simulation computes by refitting each fold's outcome model.
true_target = learned_rule_truth(learned, outcome_model)
print(f"true value of the ten fold rules: {true_target:.4f}, or {100 * true_target:.1f} per 100")
print(f"true value of the best rule:      {truth['ey_best_rule']:.4f}")

value of the learned rule: 0.7174, 95% CI 0.6907 to 0.7441
per 100 patients: 71.7 stay home, 95% CI 69.1 to 74.4
share of patients each fold's rule offers: 0.345 to 0.545
share the rules offer, by need score:


need score
-1 to -0.5    0.000
-0.5 to 0     0.002
0 to 0.5      0.805
0.5 to 1      1.000



true value of the ten fold rules: 0.7174, or 71.7 per 100
true value of the best rule:      0.7197


The ten rules offer navigation to almost no patient with a need score at or below 0 and to every
patient above 0.5. The best rule offers it above -0.1, so the learned rules miss some patients just
below 0, where navigation helps only a little. Their true value, 0.7174, is just below the best
rule's 0.7197, and in this sample the estimate lands on it.

The reported value belongs to these ten rules, not to one rule refitted on all patients, and not to
the best rule. To use a rule in the program, you would fit the outcome model once on all patients;
this fit does not estimate that rule's value. In this sample its true value is 0.7175 (printed in
the next section), close to the fold rules' value.

:::{dropdown} Details: when the interval can be trusted
The interval is valid in large samples when every patient had a real chance of either option, the
treatment model or the outcome model is good enough, and the fold rules settle on one rule as the
sample grows. The last condition fails when the offer has no effect, or almost none, for a share of
patients: the rules then flip at random, and the interval can be too narrow (see "Go deeper").
:::

## Check the weights behind the estimate

The correction step weights each patient whose offer matches the rule by one over their
predicted chance of that offer. The overlap check reads these weights: every patient needs a real
chance of the option the rule assigns.

In [8]:
assessment = learned.assess()
ledger = assessment.to_frame()
# Keep the checks that ran: `passed` has a pass mark, `completed` is a report to read.
ran = ledger[ledger["status"].isin(["passed", "completed"])]
print(ran[["check", "status"]].to_string(index=False))
print("needs attention:", tuple(item.name for item in assessment.attention) or "none")
print()
overlap = assessment.report("support").to_frame().iloc[0]
print(f"smallest predicted chance of the offer the rule assigns: {overlap['min_propensity']:.3f}")
print(f"largest weight: {overlap['max_ratio']:.2f}")
print(f"effective sample size: {overlap['effective_n']:.0f} of {len(frame)}")

          check    status
score_equations    passed
        support completed
nuisance_models completed
needs attention: none

smallest predicted chance of the offer the rule assigns: 0.429
largest weight: 2.33
effective sample size: 1139 of 2000


Nothing needs attention. The correction step reached its target (`score_equations passed`), and
`nuisance_models` records the fitted outcome and treatment models. Every patient had a predicted
chance of at least 0.429 of the option the rule assigns, so the largest weight is 2.33, and the
weights are worth 1139 equally weighted patients, the effective sample size. Warning signs are a
chance near 0, a few huge weights, or an effective sample far below the number of patients whose
offer matched the rule. A clean check reads the fitted treatment model, so it cannot show that the
model is right.

## What goes wrong when a rule is scored on the patients who chose it

The common mistake is to fit one outcome model on all patients, read off the rule, and score the
rule with the same model on the same patients. The code does that by hand, outside `cleverly`,
with the logistic model above and with a random forest, a flexible model that can memorize noise.
The simulation then computes the true value of each rule.

In [9]:
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier

from cleverly.datasets import treatment_rule_value
from cleverly.learners import thread_limit

inputs = ["offer", "need", "lives_alone"]


def with_offer(data, offer):
    return data[["need", "lives_alone"]].assign(offer=float(offer))[inputs]


forest = RandomForestClassifier(n_estimators=200, min_samples_leaf=5, random_state=17, n_jobs=1)
rows = []
with thread_limit():
    for label, model in (("logistic, as above", outcome_model), ("random forest", forest)):
        fitted = clone(model).fit(frame[inputs], frame["home_30"])

        def chance(data, offer, fitted=fitted):
            return fitted.predict_proba(with_offer(data, offer))[:, 1]

        def rule(data, chance=chance):
            return (chance(data, 1) > chance(data, 0)).astype(float)

        offered = rule(frame)
        score = np.where(offered == 1, chance(frame, 1), chance(frame, 0)).mean()
        rows.append(
            {
                "model": label,
                "share offered": offered.mean(),
                "in-sample score": score,
                "true value": treatment_rule_value(rule),  # simulation only
            }
        )
scored = pd.DataFrame(rows)
print(scored.round(4).to_string(index=False))

             model  share offered  in-sample score  true value
logistic, as above         0.4475           0.7303      0.7175
     random forest         0.4590           0.7746      0.6979


Both rules look better on the patients who chose them than they are. The logistic rule scores
0.7303 against a true value of 0.7175, and the random forest rule scores 0.7746 against 0.6979. The
forest fits noise: it offers navigation to patients who happened to do well with it and withholds
it from patients who happened to do well without it, then counts that luck as the rule's success.
The in-sample score even ranks the worse rule first.

Across 400 simulated samples (see "Go deeper")
the forest's in-sample score was too high by 0.0871 on average, and the logistic one by
0.0036. The same patients both chose the rule and scored it, and no standard error allows for
that, so such a score has no honest interval. The fit above avoids the problem, because each rule is
scored only on patients it never saw.

## When capacity forces a lottery

Suppose staff can cover only about a quarter of the patients. The program then offers navigation
by lottery: each patient with a need score above 0 gets the offer with chance one half, and no
one else does. Because the program sets those chances itself, they are known exactly, and
`Stochastic(..., density_kind="known")` declares them. `RegimeMean` asks for the share who stay
home under each option: the lottery, and no one offered. With simple models
that cannot memorize their training rows, the fit uses every patient for every model
(`CrossFitting(enabled=False)`).

In [10]:
from cleverly import RegimeMean
from cleverly.interventions import Static, Stochastic


# For your own program, return the chances your lottery uses: one row per patient, with the
# chance of no offer and then the chance of an offer.
def lottery_chances(data):
    offer = np.where(data["need"] > 0, 0.5, 0.0)
    return np.column_stack([1 - offer, offer])


plans = (
    Static(0, name="offer to none"),
    Stochastic(lottery_chances, name="lottery", density_kind="known"),
)
in_sample = replace(method, cross_fitting=CrossFitting(enabled=False))
options = study.identify(RegimeMean(plans, reference="offer to none")).estimate(method=in_sample)
print(options.summary())

                            Effect estimate (TMLE)
Question:      effect of offer on       Rows:                            2,000
               home_30; each row is     Treated share:                   0.530
               defined below            Out-of-fold predictions:            no
Outcome:       home_30 (binary)         Outcome model:     logistic regression
Adjusted for:  need, lives_alone                        after polynomial terms
Method:        targeted maximum         Treatment model:   logistic regression
               likelihood (TMLE)        Probability limits:  [0.01471, 0.9853]
                           estimate   std. error           95% CI    p-value
------------------------------------------------------------------------------
ey_regime[offer to none]      0.668        0.016   [0.637, 0.698]    <0.0001
ey_regime[lottery]            0.694        0.012   [0.670, 0.718]    <0.0001
What each row estimates
                           meaning
---------------------------

With no offers, 0.668 of patients would stay home for 30 days; under the lottery, 0.694. Each
p-value tests only whether a share is zero. The simultaneous bands cover both rows at once, so they
are wider than the intervals, and `cleverly` marks them experimental. Warning [1] concerns flexible
models, which this fit does not use.

In [11]:
names = ["ey_regime[lottery]", "ey_regime[offer to none]"]
# psi holds the named estimates in order; .psi is an estimate's value and .ci its interval.
gain = options.contrast(lambda psi: psi[0] - psi[1], names, name="lottery minus none")
low, high = gain.ci
print(f"lottery minus no offers: {gain.psi:+.4f}, 95% CI {low:+.4f} to {high:+.4f}")
print(f"per 100 patients: {100 * gain.psi:+.1f}, 95% CI {100 * low:+.1f} to {100 * high:+.1f}")
options_assessment = options.assess()
print("needs attention:", tuple(item.name for item in options_assessment.attention) or "none")
lottery_overlap = options_assessment.report("support").to_frame().set_index("regime")
print("largest weight, by option:", lottery_overlap["max_ratio"].round(2).to_dict())
print()

# simulation only, to the end of the cell: the true gain
true_lottery = treatment_rule_value(lambda data: lottery_chances(data)[:, 1])
true_gain = true_lottery - truth["ey_none"]
print(f"true gain: {true_gain:+.4f}, or {100 * true_gain:+.1f} per 100")

lottery minus no offers: +0.0262, 95% CI +0.0108 to +0.0416
per 100 patients: +2.6, 95% CI +1.1 to +4.2
needs attention: none
largest weight, by option: {'offer to none': 3.19, 'lottery': 2.26}

true gain: +0.0335, or +3.3 per 100


The lottery raises the share who stay home by 2.6 per 100 patients against no offers (95% CI 1.1
to 4.2), and the true gain is 3.3. `options.contrast(...)` turns the two shares into a difference,
with a standard error that allows for both coming from the same patients. The lottery reaches only
patients with need above 0, where the offer mostly helps, so it gains more than offering navigation
to everyone would (0.0241 in truth, from the data step). The weight check finds nothing that needs
attention, and the largest weight is 3.19.

## How much could a missing confounder move the lottery's gain?

The robustness value asks how strong a missing confounder would have to be to erase the
lottery's gain. It needs the gain as a named estimate, so the cell refits it with
`RegimeContrast`, which reports each option's difference from the reference directly.

In [12]:
from cleverly import RegimeContrast

lottery_fit = study.identify(RegimeContrast(plans, reference="offer to none")).estimate(
    method=in_sample
)
LOTTERY = "ate_regime[lottery vs offer to none]"
sensitivity = lottery_fit.sensitivity
robustness = sensitivity.robustness_value(estimand=LOTTERY)
print(f"robustness value: {robustness['rv']:.3f} ({robustness['rva']:.3f} at the 95% limit)")
# For your own data, benchmark each recorded confounder that the lottery does not read. The
# lottery reads need, so a benchmark cannot leave it out.
benchmark = sensitivity.benchmark(covariates=("lives_alone",), estimand=LOTTERY)
bound = sensitivity.omitted_confounding(estimand=LOTTERY, benchmark=benchmark)
print(
    f"a confounder as strong as lives_alone: largest shift {bound.bias:.4f}, "
    f"95% limits {bound.ci_lower:.4f} to {bound.ci_upper:.4f}"
)

robustness value: 0.076 (0.039 at the 95% limit)
a confounder as strong as lives_alone: largest shift 0.0008, 95% limits 0.0124 to 0.0400


A missing confounder has two strengths, each a share from 0 to 1: the share of the outcome's
remaining variation it explains, and how much it would change the weights that correct for who
got an offer. The robustness value, 0.076, is a floor: a confounder weaker than that on both sides
cannot erase the gain, and 0.039 on both sides is enough to pull the end of the interval to zero.
One that is strong on a single side can erase it with the other side lower. A confounder as strong
as `lives_alone` would shift the gain by at most 0.0008, leaving its limits at 0.0124 to 0.0400.
That is a weak benchmark, because living alone moves the offer and the outcome only a little.
Whether something like unrecorded support at home could reach 0.039 is a judgement about the
program.

:::{dropdown} Details: the limits of the bound
The bound reads the lottery's gain, under the worst case in which the confounder's two effects
align. No simulation study covers it for a lottery, so read it as a description. The check cannot
leave out `need`, because the lottery reads it. `cleverly` has no such bound for the learned rule's
value.
:::

## What we can and can't conclude

For the program: a rule learned from these records, which offers navigation mainly to patients
with a need score above 0, would have 71.7% of patients stay home for 30 days (95% interval 69.1%
to 74.4%). That value belongs to the ten fold rules the fit learned, not to one final rule. If staff
can cover only a quarter of patients, a lottery among those with a need score above 0 raises the
share by 2.6 per 100 against no offers (95% interval 1.1 to 4.2). Both answers hold for adults sent
home from a participating hospital, under the three conditions and with overlap.

Neither answer says what the best possible rule achieves, and the rule's interval can be too
narrow if the offer has no effect for a share of patients. The page could not bound how far a
missing confounder could move the rule's value, because `cleverly` has no such bound for a learned
rule. One simulated sample does not show how the method performs in general.

## Go deeper

### How we know the method works

Simulation studies check each method over many repeated samples.

| part of the page | simulation study | its data |
| --- | --- | --- |
| the learned rule | [learned-rule CV-TMLE](../technical-reference/method-evidence/learned-rule-cvtmle.md) | another law of the same form: a need-like score, a yes-or-no characteristic, and an effect that changes steadily with the score, at 2,000 rows with ten folds and this page's outcome model. Its intervals covered 0.9465 and 0.9413 of the samples at two such laws |
| the learned rule, at its limit | [learned-rule CV-TMLE at the boundary](../technical-reference/method-evidence/learned-rule-cvtmle-boundary.md) | laws where the offer has no effect for anyone, or a weak one. The interval covered only 0.8938 and 0.8948 |
| the lottery | [stochastic point-treatment regimes](../technical-reference/method-evidence/stochastic-point-treatment-regimes.md) | another law, with one three-level characteristic, compared with `lmtp`, another package that fits the same estimate in R |

The lottery study gives the method the true chances of the real offer, while this page fits them.
The [technical entry](../technical-reference/point-treatment-tmle.md#validation-issues-special-to-this-method)
lists the method's evidence and limits.

A simulation we ran for this page redraws the data 400 times at the page's size and refits each
sample with the page's settings. It checks the page's readings and does not validate the method.
Its script and log are in the `cleverly` repository at
`reviews/notebook-review/probes/treatment-rules/`. The learned rule's interval covered the true
value of its fold rules in 0.9475 of the samples, and its standard error matched the spread of
the errors (ratio 0.9918). The lottery's interval covered 0.9375.

### Try to break the result

These checks change the data in a known way and refit each fit 20 times:

- `placebo` swaps in random offers that mimic the real ones. The lottery's gain should then be
  near zero, and every rule is worth the plain share who stayed home, 0.6695;
- `random_common_cause` adds a pure-noise column, so the estimate should barely move;
- `subset` refits on random 70% samples, so the refits should spread about as much as the
  standard error predicts.

None can detect a missing confounder. For the learned rule, the noise and subset refits keep the
rule each patient got and the ten folds, so they rescore the same rules rather than learn new
ones.

In [13]:
print(f"share who stayed home: {frame['home_30'].mean():.4f}")
columns = [
    "test",
    "original",
    "refuted_mean",
    "refuted_sd",
    "statistic_name",
    "statistic",
    "verdict",
]
refutations = {}
for label, fit, name in (
    ("lottery's gain", lottery_fit, LOTTERY),
    ("learned rule's value", learned, "ey_learned_rule[learned rule]"),
):
    refutations[label] = fit.diagnostics.refute(
        estimand=name, tests=("placebo", "subset", "random_common_cause"), random_state=61
    )
    print(label)
    print(refutations[label].to_frame()[columns].round(4).to_string(index=False))

share who stayed home: 0.6695


lottery's gain
               test  original  refuted_mean  refuted_sd                  statistic_name  statistic     verdict
            placebo    0.0262        0.0013      0.0067                               t     0.8566 descriptive
             subset    0.0262        0.0263      0.0048               scatter ratio rho     0.9421 descriptive
random_common_cause    0.0262        0.0264      0.0003 paired shift in standard errors     0.0230 descriptive


learned rule's value
               test  original  refuted_mean  refuted_sd                  statistic_name  statistic     verdict
            placebo    0.7174        0.6692      0.0139                               t    -0.1006 descriptive
             subset    0.7174        0.7161      0.0081               scatter ratio rho     0.9116 descriptive
random_common_cause    0.7174        0.7175      0.0005 paired shift in standard errors     0.0033 descriptive


Every check behaves as expected. With random offers, the lottery's gain averages 0.0013, near zero,
and the rules are worth 0.6692 on average, near the plain share of 0.6695. The noise column barely
moves either estimate. The subset refits spread about as the standard error predicts: their
scatter ratio, the spread divided by the predicted spread, is 0.94 and 0.91, where 1 is ideal and
20 refits leave wide room around it. The verdict reads `descriptive`: `cleverly` gives a pass or
fail only when a simulation study backs the check for this kind of fit, and here none does, so you
judge the numbers.

A stronger test replaces the outcome with simulated ones whose true gain we know, refits the
lottery's gain 20 times, and checks how close the estimates come. `dummy_outcome` draws outcomes
that ignore the patients and the offer, so the true gain is zero. `simulated_outcome` draws
outcomes that depend on both, so the true gain is known. `plasmode` also redraws the patients
from the data and their offers from the fitted treatment model. These tests have no simulated
truth for a learned rule yet.

In [14]:
from cleverly.validation import BernoulliAdjustmentOutcome

# Each generated outcome is a yes-or-no draw from a logistic model of the offer and the
# characteristics. effect sets the offer's effect (log-odds), adjustment_scale how strongly the
# characteristics move it.
generated = {
    "dummy_outcome": BernoulliAdjustmentOutcome(effect=0.0, adjustment_scale=0.0),
    "simulated_outcome": BernoulliAdjustmentOutcome(effect=0.5),
    "plasmode": BernoulliAdjustmentOutcome(effect=0.5),
}
generated_tests = {}
for name, declaration in generated.items():
    keyword = "dummy_outcome" if name == "dummy_outcome" else "simulated_outcome"
    report = lottery_fit.diagnostics.refute(
        estimand=LOTTERY,
        tests=(name,),
        n_replicates=20,
        random_state=61,
        **{keyword: declaration},
    )
    test = generated_tests[name] = report.tests[0]
    bias = test.readings[0]
    print(
        f"{name:18s} {test.n_replicates} refits: true gain {test.declared_effect:.4f}, "
        f"mean estimate {test.mean:.4f}; bias in SDs {bias.estimate:.3f}, interval "
        f"{bias.lower:.2f} to {bias.upper:.2f}, pass zone {bias.band[0]} to {bias.band[1]}; "
        f"verdict {test.verdict}"
    )

dummy_outcome      20 refits: true gain 0.0000, mean estimate -0.0006; bias in SDs -0.078, interval -0.52 to 0.36, pass zone -0.2 to 0.2; verdict descriptive


simulated_outcome  20 refits: true gain 0.0261, mean estimate 0.0271; bias in SDs 0.136, interval -0.30 to 0.57, pass zone -0.2 to 0.2; verdict descriptive


plasmode           20 refits: true gain 0.0261, mean estimate 0.0248; bias in SDs -0.176, interval -0.61 to 0.26, pass zone -0.2 to 0.2; verdict descriptive


Each mean estimate lands near its true gain, such as 0.0271 against 0.0261 for
`simulated_outcome`. The bias is measured in standard deviations of the estimates, and a pass
needs its whole interval inside -0.2 to 0.2. At 20 refits the interval is wide, so it shows only
that the bias is not large, and no simulation study backs a verdict for this fit.

## Where to go next

The [interventions example](interventions.ipynb) compares a fixed screening rule and other plans
with offering to everyone. The
[point-treatment TMLE reference](../technical-reference/point-treatment-tmle.md#learned-rules)
states the conditions of the learned-rule value, and the
[estimands guide](../user-guide/estimands.md#learned-treatment-rules) shows what a learned-rule
result holds. The [examples index](index.md) lists every example.